# VecDB Embedding & Ingest Patterns
Explore hosted embeddings, manual ingest, and batching workflows in Oracle VecDB.


## 1. Scenario Overview
This notebook demonstrates:
1. Generating embeddings client-side with SentenceTransformers.
2. Uploading vectors via `upsert_vectors` (bring-your-own mode).
3. Creating an auto-embedding table using `embed_params` (server-side generation).
4. Casting batch documents to vectors with VecDB's `generate_embedding`.
5. Comparing ingest strategies and cleanup steps.


## Architecture at a Glance

```text
One sample text set, three embedding surfaces

                              +----------------------------+
                              |    Sample text snippets    |
                              +-------------+--------------+
                                            |
              +-----------------------------+------------------------------+
              |                             |                              |
              v                             v                              v
+---------------------------+   +---------------------------+   +---------------------------+
| A. BYOV ingest            |   | B. Auto-embedding table   |   | C. Batch endpoint         |
+---------------------------+   +---------------------------+   +---------------------------+
| local SentenceTransformer |   | VecDB hosted model        |   | VecDB hosted model        |
| creates vectors           |   | configured in embed_params|   | called directly           |
+-------------+-------------+   +-------------+-------------+   +-------------+-------------+
              |                             |                              |
              v                             v                              v
+---------------------------+   +---------------------------+   +---------------------------+
| upsert ID + dense_vector  |   | upsert ID + raw text      |   | generate_embedding(inputs)|
| into EMBED_BYOV_DEMO      |   | into EMBED_AUTO_DEMO      |   | returns vectors only      |
+-------------+-------------+   +-------------+-------------+   +-------------+-------------+
              |                             |                              |
              v                             v                              v
+---------------------------+   +---------------------------+   +---------------------------+
| query with local vector   |   | query with text           |   | inspect dimensions/sample |
+---------------------------+   +---------------------------+   +---------------------------+
```

The notebook compares where embeddings are produced, what payload shape is sent to VecDB, and whether rows are persisted.


## What to Validate

- The BYOV path should show vectors generated locally with SentenceTransformers and then inserted with metadata.
- The auto-embedding path should show rows inserted as text/metadata while VecDB owns vector generation.
- The batch endpoint should return embeddings without creating or modifying table rows.


### Prerequisites
- `.env` with `VECDB_REST_URL`, `VECDB_USERNAME`, `VECDB_PASSWORD`.
- Python 3.10+ with `oracle-vecdb`, `python-dotenv`, `pandas`, `sentence-transformers`.
- Optional: HF token if you want faster downloads.


In [ ]:
%pip install -U oracle-vecdb python-dotenv pandas sentence-transformers


## 2. Load Environment & Connect
Authenticate once using the `.env` credentials so the remaining sections share a single Oracle VecDB client.


In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import OracleVecDB, Configuration

print("Loading environment variables and preparing VecDB client...")
load_dotenv()

resolved_host = os.getenv('VECDB_REST_URL')
resolved_user = os.getenv('VECDB_USERNAME') or os.getenv('VECDB_USER')
resolved_password = os.getenv('VECDB_PASSWORD')
resolved_access_token = os.getenv('VECDB_ACCESS_TOKEN')
print(f"Resolved REST endpoint: {resolved_host}")
print(f"Resolved user: {resolved_user}")

config_kwargs = {"rest_url": resolved_host}
if resolved_access_token:
    config_kwargs["access_token"] = resolved_access_token
else:
    config_kwargs["username"] = resolved_user
    config_kwargs["password"] = resolved_password
config = Configuration(**config_kwargs)
if os.getenv('VECDB_SELF_SIGNED_SSL', 'false').lower() == 'true':
    config.verify_ssl = False
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print('Configured client to bypass SSL verification for self-signed certificates.')

vecdb = OracleVecDB(config)
auth_method = 'bearer token' if config.access_token else 'username/password'
print('Connected to', config.rest_url)
print('Auth method:', auth_method)
print('VecDB client ready for vector operations.')


In [ ]:
print('Configuring table names and sample documents for the demo...')
BYOV_TABLE = os.getenv('EMBED_BYOV_TABLE', 'EMBED_BYOV_DEMO')
AUTO_TABLE = os.getenv('EMBED_AUTO_TABLE', 'EMBED_AUTO_DEMO')
BATCH_TEXTS = [
    'Our vision is to unify customer data for better personalization.',
    'The marketing team plans a Q3 campaign focused on loyalty.',
    'Partners provide SLA agreements with quarterly reviews.',
    'We are preparing a webinar to highlight new analytics features.'
]
print(f'BYOV table name -> {BYOV_TABLE}')
print(f'Auto-embedding table name -> {AUTO_TABLE}')
print(f'Prepared {len(BATCH_TEXTS)} marketing snippets for ingestion.')


## 3. Define Constants
Name the demo tables and seed documents that fuel each ingest and search pattern showcased below.


In [ ]:
MODEL_NAME = os.getenv('EMBED_MODEL_NAME', 'TEXT_EMBEDDING_MODEL')
print(f'VecDB auto-embedding model target: {MODEL_NAME}')


### Pattern Comparison

The table below is a static comparison of the three embedding paths used in this notebook. It does not call VecDB; the live VecDB outputs begin in the implementation cells that follow.


In [ ]:
import pandas as pd
from IPython.display import display

pattern_comparison = pd.DataFrame([
    {
        'pattern': 'BYOV ingest',
        'who_generates_vectors': 'Notebook / local model',
        'payload_shape': 'id + dense_vector + metadata',
        'vecdb_method': 'upsert_vectors',
        'best_for': 'Full control over model choice and vectors',
    },
    {
        'pattern': 'Auto-embedding table',
        'who_generates_vectors': 'VecDB hosted model',
        'payload_shape': 'id + metadata text',
        'vecdb_method': 'create_vector_table(embed_params) + upsert_vectors',
        'best_for': 'Simpler ingest when a hosted model is available',
    },
    {
        'pattern': 'Batch embedding endpoint',
        'who_generates_vectors': 'VecDB hosted model',
        'payload_shape': 'list of input strings',
        'vecdb_method': 'generate_embedding',
        'best_for': 'Previewing embeddings or powering custom pipelines',
    },
])

display(pattern_comparison)


## 4. Bring Your Own Embeddings
Walk through a fully client-managed embedding workflow with SentenceTransformers generating vectors locally.


In [ ]:
from sentence_transformers import SentenceTransformer
HF_MODEL_NAME = os.getenv('EMBED_HF_MODEL_NAME', 'sentence-transformers/all-MiniLM-L12-v2')
print(f'Loading local embedding model via SentenceTransformers: {HF_MODEL_NAME}')
local_model = SentenceTransformer(HF_MODEL_NAME)
print('Local embedding model ready for BYOV workflow.')


### 4a. Create & Seed BYOV Table
Reset the target table to avoid conflicts, then push locally generated vectors to VecDB. The DataFrame preview validates the payload.


In [ ]:
from uuid import uuid4
import pandas as pd

def query_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, tuple):
        return list(response)
    return getattr(response, "items", None) or getattr(response, "matches", None) or []


def result_metadata(item):
    return item.get("metadata", {}) if isinstance(item, dict) else getattr(item, "metadata", {})


def result_distance(item):
    if isinstance(item, dict):
        return item.get("distance")
    return getattr(item, "distance", getattr(item, "score", None))


print('Preparing BYOV table workflow in VecDB...')
existing_tables = vecdb.list_vector_tables().items or []
if any(tbl.table_name == BYOV_TABLE for tbl in existing_tables):
    print(f"Found existing table {BYOV_TABLE}; dropping for a clean start.")
    vecdb.drop_vector_table(name=BYOV_TABLE)

print('Creating BYOV table configured for manual embeddings...')
vecdb.create_vector_table(
    name=BYOV_TABLE,
    comment='BYOV embedding table demo',
    annotations={'TEXT': 'string', 'SOURCE': 'string'},
)

print('Seeding BYOV table with locally generated embeddings...')
seed_rows = []
for text in BATCH_TEXTS:
    seed_rows.append({
        'id': str(uuid4()),
        'dense_vector': local_model.encode(text, normalize_embeddings=False).tolist(),
        'metadata': {'TEXT': text, 'SOURCE': 'sentence-transformers'},
    })
vecdb.upsert_vectors(table_name=BYOV_TABLE, vectors=seed_rows)
print(f"Inserted {len(seed_rows)} rows into {BYOV_TABLE}.")
print('Previewing BYOV seed rows below.')
print('BYOV table ready for semantic querying.')

pd.DataFrame([row['metadata'] for row in seed_rows])


### 4b. Query BYOV Table
Issue a semantic vector query against the BYOV data to verify the similarity search results.


In [ ]:
print(f"Querying {BYOV_TABLE} with a semantic vector...")
query_vector = local_model.encode('Highlight customer loyalty initiatives').tolist()
query_results = vecdb.query(
    table_name=BYOV_TABLE,
    query_by={'vector': query_vector},
    include_vectors=False,
    top_k=3
)
matches = query_items(query_results)
if not matches:
    print('No results returned. Verify the BYOV table contains data.')
    pd.DataFrame([], columns=['TEXT', 'distance'])
else:
    print(f'Retrieved {len(matches)} matches from {BYOV_TABLE}.')
    print('Top results from BYOV table:')
    display_rows = []
    for item in matches:
        metadata = result_metadata(item)
        distance = result_distance(item)
        snippet = metadata.get('TEXT') or metadata.get('BODY') or metadata.get('TITLE') or 'Untitled'
        snippet_str = snippet if isinstance(snippet, str) else str(snippet)
        if distance is not None:
            print(f"- {snippet_str[:80]} (distance: {distance:.3f})")
        else:
            print(f"- {snippet_str[:80]}")
        display_rows.append({'TEXT': snippet_str, 'distance': distance})
    print('Previewing top matches DataFrame below.')
    pd.DataFrame(display_rows)


## 5. Auto-Embedding Table
Demonstrate a VecDB-managed pipeline where the service hosts the embedding model and ingests plain text rows.


In [ ]:
from oracle_vecdb.vecdb_errors import VecDBError
print('Preparing auto-embedding demo table in VecDB...')
auto_table_created = False
try:
    vecdb.create_vector_table(
        name=AUTO_TABLE,
        comment='Auto-embedding demo table',
        annotations={'DOC_ID': 'string', 'BODY': 'string'},
        embed_params={'model': MODEL_NAME, 'embed_metadata_jsonpath': 'BODY'},
    )
    auto_table_created = True
    print('Created auto-embed table:', AUTO_TABLE)
    print(f'Auto table {AUTO_TABLE} configured to use model {MODEL_NAME}.')
except VecDBError as exc:
    print('Auto-embedding setup failed:', exc)
    print('Set EMBED_MODEL_NAME to a model available in your VecDB instance or skip this section.')


In [ ]:
print([m.model_name for m in vecdb.list_models().items or []])

### 5a. Ingest Without Client Vectors
Insert raw sample text snippets and let VecDB populate embeddings automatically via `embed_params`.


In [ ]:
from uuid import uuid4

if auto_table_created:
    auto_rows = [
        {'id': str(uuid4()), 'metadata': {'DOC_ID': f'DOC-{i}', 'BODY': text}}
        for i, text in enumerate(BATCH_TEXTS)
    ]
    print('Streaming documents into auto-embedding table for server-side embedding...')
    vecdb.upsert_vectors(table_name=AUTO_TABLE, vectors=auto_rows)
    print(f"Inserted {len(auto_rows)} raw documents into {AUTO_TABLE}.")
    print('Table description preview follows (shows embed_params configuration).')
    auto_desc = vecdb.describe_vector_table(name=AUTO_TABLE)
    auto_desc
else:
    print('Skipping auto-embed ingest: table not created.')


### 5b. Query Auto Table
Search the auto-embedding table with plain text and inspect the returned snippets and similarity scores.


In [ ]:
if auto_table_created:
    print('Executing text-only semantic query against the auto-embedding table...')
    auto_query = vecdb.query(
        table_name=AUTO_TABLE,
        query_by={'text': 'Quarterly partner review planning'},
        include_vectors=False,
        top_k=3
    )
    auto_matches = query_items(auto_query)
    print(f'Received {len(auto_matches)} matches from {AUTO_TABLE}.')
    print('Previewing top auto-embedding matches below.')
    pd.DataFrame([
        {
            'DOC_ID': result_metadata(r).get('DOC_ID'),
            'BODY': result_metadata(r).get('BODY'),
            'distance': result_distance(r),
        }
        for r in auto_matches
    ])
else:
    print('Skipping auto-embed query: table not created.')


## 6. Batch Embedding via VecDB
Call the `generate_embedding` endpoint to transform documents server-side without persisting them.


In [ ]:
print('Requesting VecDB to generate embeddings for the batch payload...')
batch_embeddings = vecdb.generate_embedding(
    model_name=MODEL_NAME,
    inputs=BATCH_TEXTS
)
print(f"Received {len(batch_embeddings.data)} embeddings from model {MODEL_NAME}.")
print('Previewing dimensionality and first few values for validation below.')
len(batch_embeddings.data), batch_embeddings.data[0].embedding[:5]


## 7. Cleanup
Drop the demo tables so repeated runs start clean and leave no lingering artifacts in VecDB.


In [ ]:
print('Cleaning up demo artifacts from VecDB...')
vecdb.drop_vector_table(name=BYOV_TABLE)
print(f'Removed table {BYOV_TABLE}.')
vecdb.drop_vector_table(name=AUTO_TABLE)
print(f'Removed table {AUTO_TABLE}.')
print('Dropped demo tables')
